In [ ]:
!unzip mallorn-astronomical-classification-challenge.zip

import pandas as pd
train_log  = pd.read_csv('train_log.csv')
train_log = train_log.drop(
    columns=['Z_err', 'SpecType', 'English Translation']
)


import numpy as np
data_log_np = train_log.to_numpy()

split_map = {}
splits_names  = {'split_01', 'split_02' , 'split_03' , 'split_04' , 'split_05', 'split_06' , 'split_07' , 'split_08' , 'split_09', 'split_10' , 'split_11' , 'split_12' , 'split_13', 'split_14' , 'split_15' , 'split_16' , 'split_17', 'split_18' , 'split_19' , 'split_20'}


Archive:  mallorn-astronomical-classification-challenge.zip
replace sample_submission.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
!pip install catboost
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
filters = ['u','g','r','i','z','y']
quantiles = [0.25, 0.5, 0.75]
for x in splits_names :
  dta = pd.read_csv(x + '/train_full_lightcurves.csv')
  split_map[x] = dta
import numpy as np
from scipy.stats import skew, kurtosis

from scipy.stats import skew, kurtosis
import numpy as np

filters = ['u','g','r','i','z','y']
quantiles = [0.1,0.25,0.5,0.75,0.9]

N_FEATURES_PER_FILTER = 14 + len(quantiles)  # doit rester constant

def extract_features(df_obj):
    features = []
    peak_times = {}
    mean_fluxes = {}

    for f in filters:
        df_f = df_obj[df_obj['Filter'] == f]

        if len(df_f) < 3:
            features.extend([0]*N_FEATURES_PER_FILTER)
            peak_times[f] = 0
            mean_fluxes[f] = 0
            continue

        flux = df_f['Flux'].values.astype(float)
        time = df_f['Time (MJD)'].values.astype(float)
        ferr = df_f['Flux_err'].values.astype(float)

        idx_peak = np.argmax(flux)

        amp = flux.max() - flux.min()
        mean_flux = flux.mean()
        std_flux = flux.std() + 1e-6
        median_flux = np.median(flux)
        snr = np.mean(np.abs(flux) / (ferr + 1e-6))

        skew_flux = skew(flux)
        kurt_flux = kurtosis(flux)

        skew_flux = 0.0 if not np.isfinite(skew_flux) else skew_flux
        kurt_flux = 0.0 if not np.isfinite(kurt_flux) else kurt_flux

        duration = time.max() - time.min()
        n_points = len(flux)

        rise_time = time[idx_peak] - time.min()
        fall_time = time.max() - time[idx_peak]

        slope_rise = (flux[idx_peak] - flux.min()) / (rise_time + 1e-6)
        slope_fall = (flux[idx_peak] - flux[-1]) / (fall_time + 1e-6)

        slope_rise = np.clip(slope_rise, -1e3, 1e3)
        slope_fall = np.clip(slope_fall, -1e3, 1e3)

        variability = std_flux / (np.abs(mean_flux) + 1e-6)

        q = np.quantile(flux, quantiles)

        feats = [
            amp, mean_flux, std_flux, median_flux, snr,
            skew_flux, kurt_flux,
            duration, n_points,
            rise_time, fall_time,
            slope_rise, slope_fall,
            variability
        ] + list(q)

        feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)

        features.extend(feats)
        peak_times[f] = time[idx_peak]
        mean_fluxes[f] = mean_flux

    for f1, f2 in [('g','r'), ('r','i'), ('i','z')]:
        features.append(mean_fluxes.get(f1,0) - mean_fluxes.get(f2,0))

    for f1, f2 in [('g','r'), ('r','i'), ('i','z')]:
        features.append(peak_times.get(f1,0) - peak_times.get(f2,0))

    return np.array(features, dtype=np.float32)


# Générer X_train et y
X_train = []
y_train = []

for row in data_log_np:  # row contient object_id, split, target...
    df_split = split_map[row[3]]
    df_obj = df_split[df_split['object_id']==row[0]]
    feats = extract_features(df_obj)
    X_train.append(feats)
    y_train.append(row[-1])

X_train = np.array(X_train)
y_train = np.array(y_train)


In [ ]:


df_obj = df_obj.sort_values("Time (MJD)")
times = df_obj["Time (MJD)"].unique()
times = np.sort(times)
FILTERS = ['u','g','r','i','z','y']

import torch.nn as nn
import torch
from torch.utils.data import Dataset
import torch
import torch
import torch.nn as nn
import torch.nn.functional as F

def build_sequence(df_obj, z, ebv, max_len=200):
    df_obj = df_obj.sort_values("Time (MJD)")
    times = np.sort(df_obj["Time (MJD)"].unique())

    seq = []
    prev_t = times[0]

    for t in times:
        row_feats = []
        df_t = df_obj[df_obj["Time (MJD)"] == t]

        for f in FILTERS:
            df_f = df_t[df_t["Filter"] == f]
            if len(df_f) == 1:
                flux = df_f["Flux"].values[0]
                ferr = df_f["Flux_err"].values[0]
                mask = 1.0
            else:
                flux, ferr, mask = 0.0, 0.0, 0.0

            row_feats.extend([flux, ferr, mask])

        delta_t = t - prev_t
        prev_t = t

        row_feats.append(delta_t)
        row_feats.append(z)
        row_feats.append(ebv)

        seq.append(row_feats)

    seq = np.array(seq)
    length = len(seq)

    if length < max_len:
        pad = np.zeros((max_len - length, seq.shape[1]))
        seq = np.vstack([seq, pad])
    else:
        seq = seq[:max_len]
        length = max_len

    return seq, length
class MallornDataset(Dataset):
    def __init__(self, log_df, split_map, max_len=200, train=True):
        self.log_df = log_df.reset_index(drop=True)
        self.split_map = split_map
        self.max_len = max_len
        self.train = train

    def __len__(self):
        return len(self.log_df)

    def __getitem__(self, idx):
        row = self.log_df.iloc[idx]
        obj_id = row["object_id"]
        split_name = row["split"]
        z = row["Z"]
        ebv = row["EBV"]

        df_split = self.split_map[split_name]
        df_obj = df_split[df_split["object_id"] == obj_id]

        seq, length = build_sequence(df_obj, z, ebv, self.max_len)

        if self.train:
            label = row["target"]
            return (
                torch.tensor(seq, dtype=torch.float32),
                torch.tensor(length, dtype=torch.long),
                torch.tensor(label, dtype=torch.float32),
            )
        else:
            return (
                torch.tensor(seq, dtype=torch.float32),
                torch.tensor(length, dtype=torch.long),
                obj_id,
            )
class LightCurveCNNLSTM(nn.Module):
    def __init__(self, n_features, hidden_size=128, lstm_layers=1, dropout=0.3):
        super().__init__()

        self.conv1 = nn.Conv1d(n_features, 64, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(64)

        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(128)

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=hidden_size,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x, lengths):
        # x: (B,T,F)
        x = x.permute(0, 2, 1)  # (B,F,T)

        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))

        x = x.permute(0, 2, 1)  # (B,T,128)

        # Pack sequence
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        packed_out, (h_n, _) = self.lstm(packed)

        # Concat last hidden states (bidirectional)
        h_forward = h_n[-2]
        h_backward = h_n[-1]
        out = torch.cat([h_forward, h_backward], dim=1)

        out = self.dropout(out)
        out = F.relu(self.fc1(out))
        out = self.dropout(out)
        out = self.fc2(out)

        return out.squeeze(1)


In [ ]:
!pip install --upgrade torch sympy


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.metrics import f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
# -----------------------
# 1️⃣ Calcul pos_weight
# -----------------------
n_pos = np.sum(y_train == 1)
n_neg = np.sum(y_train == 0)
print("Positives:", n_pos, "Negatives:", n_neg)

pos_weight = torch.tensor([n_neg / (n_pos + 1e-6)], dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# -----------------------
# 2️⃣ Dataset + DataLoader
# -----------------------

train_log_df = pd.read_csv('train_log.csv')

train_dataset = MallornDataset(log_df=train_log_df, split_map=split_map, train=True, max_len=200)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)


# -----------------------
# 3️⃣ Modèle + optimizer
# -----------------------
n_features = 21  # flux_u,g,r,i,z,y + flux_err + mask + delta_t + Z + EBV (adapter si tu changes)
model = LightCurveCNNLSTM(n_features=n_features).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# -----------------------
# 4️⃣ Boucle d'entraînement
# -----------------------
import math
for epoch in range(25):
    model.train()
    total_loss = 0

    for xb, lengths, yb in train_loader:

        xb = torch.nan_to_num(xb, nan=0.0, posinf=0.0, neginf=0.0)
        xb = xb.to(device)
        lengths = lengths.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(xb, lengths)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

cuda
Positives: 148 Negatives: 2895
Epoch 1 Loss: 1.3227
Epoch 2 Loss: 1.3242
Epoch 3 Loss: 1.3188
Epoch 4 Loss: 1.2744
Epoch 5 Loss: 1.2500
Epoch 6 Loss: 1.2328
Epoch 7 Loss: 1.2018
Epoch 8 Loss: 1.1613
Epoch 9 Loss: 1.1405
Epoch 10 Loss: 1.1186
Epoch 11 Loss: 1.1169
Epoch 12 Loss: 1.1612


KeyboardInterrupt: 

In [ ]:

for x in splits_names :
  dta = pd.read_csv(x + '/test_full_lightcurves.csv')
  split_map[x] = dta
class MallornTestDataset(Dataset):
    def __init__(self, log_df, split_map, max_len=200):
        self.log_df = log_df.reset_index(drop=True)
        self.split_map = split_map
        self.max_len = max_len

    def __len__(self):
        return len(self.log_df)

    def __getitem__(self, idx):
        row = self.log_df.iloc[idx]
        obj_id = row["object_id"]
        split_name = row["split"]
        z = row["Z"]
        ebv = row["EBV"]

        df_split = self.split_map[split_name]
        df_obj = df_split[df_split["object_id"] == obj_id]

        seq, length = build_sequence(df_obj, z, ebv, self.max_len)

        return (
            torch.tensor(seq, dtype=torch.float32),
            torch.tensor(length, dtype=torch.long),
            obj_id
        )

test_log_df = pd.read_csv('test_log.csv')
test_dataset = MallornTestDataset(test_log_df, split_map, max_len=200)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
model.eval()
all_probs = []
all_ids = []

with torch.no_grad():
    for xb, lengths, obj_ids in test_loader:
        xb = xb.to(device)
        lengths = lengths.to(device)

        logits = model(xb, lengths)
        probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.extend(probs)
        all_ids.extend(obj_ids)


In [ ]:
# Si tu as déjà trouvé le best_thresh sur la validation
preds = (np.array(all_probs) > 0.677).astype(int)
submission = pd.DataFrame({
    "object_id": all_ids,
    "prediction": preds
})

submission.to_csv("submission_lsm.csv", index=False)
print("Fichier submission.csv généré !")


Fichier submission.csv généré !


In [ ]:
print(len(preds))

7135


In [ ]:
print(sum(preds))

906
